In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 22,
    'font.weight': 'normal',
    'axes.titleweight': 'bold'
})

In [ ]:
data_path = '/content/drive/My Drive/volleypred/data/'
plot_path = '/content/drive/My Drive/volleypred/plots/'

In [ ]:
df = pd.read_csv(data_path + 'final_df.csv')
df.head()

In [ ]:
# VISUALIZATION 1

In [ ]:
def t1_won(score):
    try:
        s1, s2 = map(int, score.split(':'))
        return s1 > s2
    except: return False

df['T1_Won'] = df['Score'].apply(t1_won)

In [ ]:
selected_teams = {
    'ZAKSA Kędzierzyn-Koźle': 'ZAKSA',
    'Jastrzębski Węgiel': 'Jastrzębski Węgiel',
    'Asseco Resovia': 'Asseco Resovia',
    'PGE Skra Bełchatów': 'PGE Skra Bełchatów'
}


In [ ]:
team_results = []

for team in selected_teams:
    t1 = df[df['Team_T1'] == team][['Season', 'T1_Won']].rename(columns={'T1_Won': 'Won'})
    t2 = df[df['Team_T2'] == team][['Season', 'T1_Won']].copy()
    t2['Won'] = t2['T1_Won'].apply(lambda x: 0 if x == 1 else 1)

    combined = pd.concat([t1, t2[['Season', 'Won']]])
    seasonal_wr = combined.groupby('Season')['Won'].mean() * 100

    for season, rate in seasonal_wr.items():
        team_results.append({'Team': team, 'Season': season, 'Win Rate (%)': rate})

plt.figure(figsize=(14, 7))
sns.set_style("whitegrid")

sns.lineplot(data=pd.DataFrame(team_results), x='Season', y='Win Rate (%)',
             hue='Team', marker='o', linewidth=2.5)

plt.axhline(50, color='red', linestyle='--', alpha=0.6, label='50% Baseline')

plt.ylabel('Win Rate (%)')
plt.xlabel('Season')
plt.xticks(rotation=45)
plt.ylim(30, 100)

plt.legend(title='Team', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(plot_path + 'team_form_path.png', dpi=300)
plt.show()
print("The plot has been saved!")

# Visualization 2

In [ ]:
stats_to_plot = {
    'Diff_Att_Kill_Perc': 'Attack Efficiency Advantage (%)',
    'Diff_Blk_Sum': 'Block Points Advantage',
    'Diff_Srv_Ace': 'Service Aces Advantage'
}

all_stats_data = []

for full_name, short_name in selected_teams.items():
    for stat_col, stat_label in stats_to_plot.items():
        t1_subset = df[df['Team_T1'] == full_name][['Season', stat_col]].copy()
        t2_subset = df[df['Team_T2'] == full_name][['Season', stat_col]].copy()
        t2_subset[stat_col] = -t2_subset[stat_col]

        combined = pd.concat([t1_subset, t2_subset])

        seasonal_avg = combined.groupby('Season')[stat_col].mean().reset_index()
        seasonal_avg['Team'] = short_name
        seasonal_avg['Metric'] = stat_label
        seasonal_avg.rename(columns={stat_col: 'Value'}, inplace=True)

        all_stats_data.append(seasonal_avg)

df_final_plot = pd.concat(all_stats_data)

fig, axes = plt.subplots(3, 1, figsize=(15, 18), sharex=True)
sns.set_style("whitegrid")

for i, (stat_col, stat_label) in enumerate(stats_to_plot.items()):
    subset = df_final_plot[df_final_plot['Metric'] == stat_label]
    sns.lineplot(data=subset, x='Season', y='Value', hue='Team', color='muted', marker='o', linewidth=2.5, ax=axes[i])

    axes[i].set_title(f'Evolution of {stat_label} (Average Advantage vs Opponents)', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Mean Difference')
    axes[i].axhline(0, color='red', linestyle='--', alpha=0.5)
    axes[i].legend(title='Team', loc='upper left')

plt.xlabel('Season')
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(plot_path + 'teams_stats_evolution.png', dpi=300)
plt.show()
print("The plot has been saved!")